# 02. Bronze ingest

Joins orders with reviews and writes them to an Iceberg v2 table partitioned by month.

## What to look at

- **Which data source** was used (OLIST or SYNTHETIC) and the row counts.
- The **status x has_review_text** table: many reviews have a score but no text; those are kept but flagged, and never embedded.
- The **partitions** table: one row per month (hidden partitioning).

In [ ]:
import sys, pathlib
_nb = next(p for p in (pathlib.Path.cwd(), pathlib.Path.cwd() / "notebooks") if (p / "_setup.py").exists())
sys.path.insert(0, str(_nb))
from _setup import *   # puts src/ on the path, loads .env, gives run_script / get_spark / sql / minio helpers
print("SAMPLE_ROWS =", common.SAMPLE_ROWS, "(None = all rows)")

In [ ]:
import data_source
orders_csv, reviews_csv, source = data_source.locate_csvs()
pd.DataFrame([("data source", source.upper()), ("orders file", orders_csv.name), ("reviews file", reviews_csv.name),
              ("SAMPLE_ROWS", common.SAMPLE_ROWS or "all rows")], columns=["item", "value"])

In [ ]:
run_script("02_bronze_ingest.py")

In [ ]:
spark = get_spark("nb02")
spark.table(T_BRONZE).select("order_id", "order_status", "order_purchase_timestamp", "review_score",
                             "review_text", "has_review_text").limit(8).toPandas()

In [ ]:
sql(spark, f"""SELECT order_status, has_review_text, count(*) AS n
               FROM {T_BRONZE} GROUP BY 1, 2 ORDER BY n DESC""")

In [ ]:
# Iceberg stores months(...) as "months since 1970-01"; shown here as a date
sql(spark, f"""SELECT add_months(DATE'1970-01-01', partition.order_purchase_timestamp_month) AS month,
                      record_count, file_count FROM {T_BRONZE}.partitions ORDER BY month""")

In [ ]:
props = sql(spark, f"SHOW TBLPROPERTIES {T_BRONZE}")
props[props.key.isin(["format-version"])]

In [ ]:
# Cleanup. Skip this cell while you are showing the Spark UI (http://localhost:4040): it closes with the session.
stop_spark()

## Open MinIO now

http://localhost:9001 (login `minioadmin` / `minioadmin`), bucket **warehouse**.

Open `warehouse/` then `bronze/reviews/`. You should see two folders: **data/** with one sub-folder per month (`order_purchase_timestamp_month=2017-01`, ...) each holding Parquet file(s), and **metadata/** with the first `*.metadata.json`, a `snap-*.avro` manifest list and a manifest `*.avro`.